<a id="0"></a>
# <div style="padding:20px;color:white;margin:0;font-size:30px;font-family:Georgia;text-align:center;display:fill;border-radius:5px;background-color:#255E58;overflow:hidden">Análise Avançada de Dados para IoT  📊📈</div>

<a id="1"></a>
# <div style="padding:20px;color:white;margin:0;font-size:30px;font-family:Georgia;text-align:center;display:fill;border-radius:5px;background-color:#254E58;overflow:hidden">Módulo 9 → Engenharia de Atributos e Modelos
</div>

> <span style='font-size:15px; font-family:Verdana;color: #c6dadfff; border-bottom: solid 3px'><b>Importando
Bibliotecas</b></span>

In [ ]:
# BIBLIOTECAS
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, accuracy_score, confusion_matrix
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import IsolationForest
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA


# CONFIGURAÇÕES DE ESTÉTICA👇🏽

# Ajustes visuais do Jupyter
from IPython.core.display import HTML
display(HTML('<style>.container { width:80% !important; }</style>'))
# pd.set_option('display.max_columns', 60)

# Configurando pandas para mostrar todas as linhas e colunas
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None) 

# Configurando pandas para não mostrar notação científica para números
pd.set_option('display.float_format', lambda x: '%.2f' % x)
# Serve para Manter os gráficos nesse mesmo Notebook
%matplotlib inline
# Ignorando alertas
import warnings
warnings.filterwarnings("ignore")

# Reprodutibilidade
np.random.seed(42)

<a id="1"></a>
<h2 style="font-family: Verdana; font-size: 25px; font-style: normal; font-weight: normal; text-decoration: none; text-transform: none; letter-spacing: 2px; color: #254E58; background-color: #ffffff;"><b> Vídeo</b> 2

In [ ]:
# GERANDO E CRIANDO OS DADOS
tempo = pd.date_range("2026-09-29", periods=24, freq="1h")
hora = tempo.hour

turno = np.where(hora < 8, "Madrugada",
         np.where(hora < 16, "Dia", "Noite"))

operacao = np.where((hora >= 8) & (hora < 18), "Ligada", "Baixa carga")
temp_ambiente = 22 + 4 * np.sin(2 * np.pi * (hora - 6) / 24)

temperatura = (
    28
    + 0.35 * temp_ambiente
    + np.where(operacao == "Ligada", 6, 0)
    + np.random.normal(0, 0.8, 24)
)

# CRIANDO UM DATAFRAME
df = pd.DataFrame({
    "data_hora": tempo,
    "hora": hora,
    "turno": turno,
    "operacao": operacao,
    "temp_ambiente": temp_ambiente,
    "temperatura_maquina": temperatura
})

print(df.head())

# GRÁFICO TEMPORAL
plt.figure(figsize=(11, 4))
plt.plot(df["data_hora"], df["temperatura_maquina"], marker="o", label="Temperatura")
plt.axvspan(df["data_hora"].iloc[8], df["data_hora"].iloc[17],
            alpha=0.15, label="Máquina ligada")

plt.title("Temperatura da Máquina + Contexto Operacional")
plt.xlabel("Tempo")
plt.ylabel("Temperatura (°C)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# MÉTRICAS ----> MÉDIA POR TURNO
media_turno = (
    df.groupby("turno")["temperatura_maquina"]
    .mean()
    .reindex(["Madrugada", "Dia", "Noite"])
)

# CRIANDO E VISUALIZANDO OS GRÁFICOS
plt.figure(figsize=(7, 4))
plt.bar(media_turno.index, media_turno.values)

plt.title("Temperatura Média por Turno")
plt.xlabel("Contexto temporal")
plt.ylabel("Temperatura média (°C)")
plt.grid(axis="y", alpha=0.3)
plt.show()

**Conclusão**: 

* _A temperatura da máquina não varia apenas por acaso; ela muda de acordo com o horário e, principalmente, com o contexto de operação_.

<a id="1"></a>
<h2 style="font-family: Verdana; font-size: 25px; font-style: normal; font-weight: normal; text-decoration: none; text-transform: none; letter-spacing: 2px; color: #254E58; background-color: #ffffff;"><b> Vídeo</b> 3

In [ ]:
# GERANDO E CRIANDO DADOS
np.random.seed(109)
datas = pd.date_range('2026-01-01', periods=240, freq='h')
df = pd.DataFrame({
    'timestamp': datas,
    'temperatura': 24 + np.sin(np.arange(240)/12)*3 + np.random.normal(0, 0.8, 240),
    'vibracao': np.abs(np.random.normal(3 + 9/10, 0.7, 240)),
    'energia_kwh': 8 + np.sin(np.arange(240)/24)*2 + np.random.normal(0, 0.6, 240),
    'umidade': 55 + np.cos(np.arange(240)/18)*8 + np.random.normal(0, 1.5, 240)
})
df['equipamento'] = np.random.choice(['EQP_01','EQP_02','EQP_03'], size=len(df))

# VISUALIZANDO OS DADOS
df.head()

> <span style='font-size:15px; font-family:Verdana;color: #c6dadfff; border-bottom: solid 3px'><b>Engenharia de Atributos</b></span>

In [ ]:
# Engenharia de atributos temporais
df = df.sort_values('timestamp')
df['hora'] = df['timestamp'].dt.hour
df['dia_semana'] = df['timestamp'].dt.dayofweek
df['energia_lag_1h'] = df['energia_kwh'].shift(1)
df['energia_media_6h'] = df['energia_kwh'].rolling(6).mean()
df['tendencia_6h'] = df['energia_kwh'] - df['energia_media_6h']
print(df[['timestamp','energia_kwh','energia_lag_1h','energia_media_6h','tendencia_6h']].tail())

df.set_index('timestamp')[['energia_kwh','energia_media_6h']].plot(figsize=(10,4))
plt.title('Atributos de janela')
plt.grid(True)
plt.show()

> <span style='font-size:15px; font-family:Verdana;color: #c6dadfff; border-bottom: solid 3px'><b>Visualizando os dados em um Diplay</b></span>

In [ ]:
styled_df = df.tail().style # Visualizando os 5 últimos registros

# Definindo a cor de fundos, a cor do texto, e a cor do DataFrame
styled_df.set_properties(**{"background-color": "#254E58", "color": "#e9c46a", "border": "1.5px solid black"})

# Modificando a cor de fundo dos cabeçalhos da tabela
styled_df.set_table_styles([
    {"selector": "th", "props": [("color", 'white'), ("background-color", "#333333")]}
])

**Conclusão**

 * O gráfico mostra que a média móvel de 6 horas reduz o ruído das leituras individuais e evidencia o comportamento geral do consumo de energia. _Assim, em vez de analisar somente o valor atual do sensor, conseguimos analisar também como ele vinha se comportando recentemente, facilitando a identificação de tendências, mudanças e possíveis anomalias_.

<a id="1"></a>
<h2 style="font-family: Verdana; font-size: 25px; font-style: normal; font-weight: normal; text-decoration: none; text-transform: none; letter-spacing: 2px; color: #254E58; background-color: #ffffff;"><b> Vídeo</b> 4

##### **Lags e histórico**

In [ ]:
# GERANDO E CRIANDO OS DADOS
tempo = pd.date_range('2026-01-01', periods=120, freq='h')
temperatura = 24 + np.sin(np.arange(120)/8)*3 + np.random.normal(0, 0.8, 120)
umidade = 55 + np.cos(np.arange(120)/10)*5 + np.random.normal(0, 1.5, 120)
df_lag = pd.DataFrame({'timestamp': tempo, 'temperatura': temperatura, 'umidade': umidade})
df_lag['media_movel_temp'] = df_lag['temperatura'].rolling(6).mean()
df_lag['lag_temp'] = df_lag['temperatura'].shift(1)
df_lag['alerta_temp'] = df_lag['temperatura'] > 28
print(df_lag.head())
print('\nResumo:')
print(df[['temperatura','umidade']].describe())
plt.figure(figsize=(10,4))
plt.plot(df_lag['timestamp'], df_lag['temperatura'], label='temperatura')
plt.plot(df_lag['timestamp'], df_lag['media_movel_temp'], label='média móvel')
plt.xticks(rotation=45)
plt.title('Série temporal de temperatura - IoT')
plt.xlabel('Tempo')
plt.ylabel('Temperatura')
plt.legend()
plt.tight_layout()
plt.show()

> <span style='font-size:15px; font-family:Verdana;color: #c6dadfff; border-bottom: solid 3px'><b>Estatísticas Descritiva</b></span>

In [ ]:
styled_df = df_lag.describe().style # Visualizando os 5 últimos registros

# Definindo a cor de fundos, a cor do texto, e a cor do DataFrame
styled_df.set_properties(**{"background-color": "#254E58", "color": "#e9c46a", "border": "1.5px solid black"})

# Modificando a cor de fundo dos cabeçalhos da tabela
styled_df.set_table_styles([
    {"selector": "th", "props": [("color", 'white'), ("background-color", "#333333")]}
])

> <span style='font-size:15px; font-family:Verdana;color: #c6dadfff; border-bottom: solid 3px'><b>Visualizando os Dados</b></span>

In [ ]:
styled_df = df_lag.head(8).style # Visualizando os 5 últimos registros

# Definindo a cor de fundos, a cor do texto, e a cor do DataFrame
styled_df.set_properties(**{"background-color": "#254E58", "color": "#e9c46a", "border": "1.5px solid black"})

# Modificando a cor de fundo dos cabeçalhos da tabela
styled_df.set_table_styles([
    {"selector": "th", "props": [("color", 'white'), ("background-color", "#333333")]}
])

**Conclusão**
* O gráfico mostra que a temperatura apresenta um padrão periódico ao longo do tempo, com oscilações regulares entre valores mais baixos e mais altos. _A média móvel ajuda a visualizar esse padrão de forma mais clara, reduzindo as pequenas variações das leituras do sensor._

   * A média móvel suaviza as leituras do sensor e facilita a identificação de tendências e padrões temporais na temperatura.

<a id="1"></a>
<h2 style="font-family: Verdana; font-size: 25px; font-style: normal; font-weight: normal; text-decoration: none; text-transform: none; letter-spacing: 2px; color: #254E58; background-color: #ffffff;"><b> Vídeo</b> 5

In [ ]:
# GERANDO E CRIANDO DADOS
np.random.seed(42)
n = 220

tempo = pd.date_range("2026-09-29 08:00", periods=n, freq="1min")

temperatura = [30]
for _ in range(1, n):
    temperatura.append(
        0.92 * temperatura[-1] + 2.4 + np.random.normal(0, 0.7)
    )

df = pd.DataFrame({
    "tempo": tempo,
    "temperatura": temperatura
})

df["alvo"] = df["temperatura"].shift(-1)
df["temp_atual"] = df["temperatura"]
df["futuro_vazado"] = df["alvo"]
df = df.dropna()


# TREINO E TESTE
corte = int(len(df) * 0.8)
treino, teste = df.iloc[:corte], df.iloc[corte:]


def prever(colunas):
    modelo = LinearRegression()
    modelo.fit(treino[colunas], treino["alvo"])
    return modelo.predict(teste[colunas])


def metricas(real, pred):
    mae = mean_absolute_error(real, pred)
    rmse = np.sqrt(mean_squared_error(real, pred))
    return mae, rmse


# MODELOS
pred_ok = prever(["temp_atual"])
pred_leak = prever(["temp_atual", "futuro_vazado"])

mae_ok, rmse_ok = metricas(teste["alvo"], pred_ok)
mae_leak, rmse_leak = metricas(teste["alvo"], pred_leak)

print("SEM DATA LEAKAGE")
print(f"MAE: {mae_ok:.3f} → RMSE: {rmse_ok:.3f}")

print("\nCOM DATA LEAKAGE")
print(f"MAE: {mae_leak:.3f} → RMSE: {rmse_leak:.3f}")


# GRÁFICO 1 - TREINO x TESTE
plt.figure(figsize=(12, 4))

plt.plot(treino["tempo"], treino["temperatura"],
         color="blue", label="Treino")

plt.plot(teste["tempo"], teste["temperatura"],
         color="orange", label="Teste")

plt.axvline(teste["tempo"].iloc[0],
            color="blue", linestyle="--",
            linewidth=2, label="Corte temporal")

plt.title("Divisão correta: passado para treino, futuro para teste")
plt.xlabel("Tempo")
plt.ylabel("Temperatura (°C)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


# GRÁFICO 2 - DATA LEAKAGE
plt.figure(figsize=(12, 5))

plt.plot(teste["tempo"], teste["alvo"],
         color="blue", linewidth=4,
         alpha=0.5, label="Real")

plt.plot(teste["tempo"], pred_ok,
         color="orange", linestyle="--",
         linewidth=2.5, label="Sem leakage")

plt.plot(teste["tempo"], pred_leak,
         color="red", linestyle=":",
         linewidth=3, marker="o",
         markersize=4, label="Com leakage")

plt.title("Efeito do Data Leakage")
plt.xlabel("Tempo")
plt.ylabel("Temperatura do próximo minuto")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**A conclusão é*:

- Sem leakage: MAE = 0,559 e RMSE = 0,746
- Com leakage: MAE = 0 e RMSE = 0

*O modelo sem leakage é o resultado que realmente deve ser considerado*. Um MAE de 0,559 significa que o modelo erra, em média, cerca de 0,56 °C. O RMSE de 0,746 °C indica que, considerando também os erros maiores, o erro típico fica abaixo de 1 °C.

<a id="1"></a>
<h2 style="font-family: Verdana; font-size: 25px; font-style: normal; font-weight: normal; text-decoration: none; text-transform: none; letter-spacing: 2px; color: #254E58; background-color: #ffffff;"><b> Vídeo</b> 6

> <span style='font-size:15px; font-family:Verdana;color: #c6dadfff; border-bottom: solid 3px'><b>Exemplo 1</b></span>

In [ ]:
# GERANDO E CRIANDO DADOS
horas = pd.date_range('2026-01-01', periods=72, freq='h')
talhoes = ['T1','T2','T3','T4']
linhas=[]
for t in talhoes:
    fator = {'T1':0, 'T2':-5, 'T3':4, 'T4':-9}[t]
    umidade = 38 + fator + 8*np.sin(np.arange(72)/24*2*np.pi) + np.random.normal(0,2,72)
    for h,u in zip(horas, umidade):
        linhas.append([h,t,u])
        
# CRIANDO O DATAFRAME
df = pd.DataFrame(linhas, columns=['timestamp','talhao','umidade_solo'])
df['risco_hidrico'] = (df['umidade_solo'] < 30).astype(int)

# VISUALIZANDO
print(df.groupby('talhao')[['umidade_solo','risco_hidrico']].mean().round(2))

# CRIANDO E VISUALIZANDO OS GRÁFICOS
pivot = df.pivot(index='timestamp', columns='talhao', values='umidade_solo')
pivot.plot(figsize=(10,4), title='Umidade do solo por talhão')
plt.ylabel('Umidade (%)')
plt.grid(True, alpha=.3)
plt.show()

# VISUALIZANDO OS RESULTADOS
print('Talhão mais crítico:', df.groupby('talhao')['risco_hidrico'].mean().idxmax())

> <span style='font-size:15px; font-family:Verdana;color: #c6dadfff; border-bottom: solid 3px'><b>Exemplo 2</b></span>

In [ ]:
# GERANDO E CRIANDO DADOS
estufas = ["E1", "E2", "E3"]
tempo = pd.date_range("2026-09-27", periods=48, freq="1h")

offset_temp = {"E1": 0, "E2": 2, "E3": 4}
offset_umid = {"E1": 0, "E2": -5, "E3": -10}

dados = []

for e in estufas:
    for dt in tempo:
        h = dt.hour

        temp = (
            24
            + 6 * np.sin(2 * np.pi * (h - 6) / 24)
            + offset_temp[e]
            + np.random.normal(0, 1)
        )

        umid = (
            72
            - 18 * np.sin(2 * np.pi * (h - 6) / 24)
            + offset_umid[e]
            + np.random.normal(0, 3)
        )

        luz = max(0, 800 * np.sin(np.pi * (h - 6) / 12)) if 6 <= h <= 18 else 0
        vent = int(temp > 28)

        dados.append([e, dt, temp, umid, luz, vent])

df = pd.DataFrame(
    dados,
    columns=[
        "estufa", "data_hora", "temperatura",
        "umidade", "luminosidade", "ventilacao_aberta"
    ]
)

# ATRIBUTOS E RISCO
df["temp_excesso"] = (df["temperatura"] - 30).clip(lower=0)
df["umid_deficit"] = (55 - df["umidade"]).clip(lower=0)
df["luz_excesso"] = (df["luminosidade"] - 700).clip(lower=0)
df["vent_fechada"] = 1 - df["ventilacao_aberta"]

df["risco"] = (
    40 * df["temp_excesso"] / 8
    + 25 * df["umid_deficit"] / 25
    + 15 * df["luz_excesso"] / 300
    + 20 * df["vent_fechada"]
).clip(0, 100)

# RESUMO
resumo = df.groupby("estufa").agg(
    temp_media=("temperatura", "mean"),
    umid_media=("umidade", "mean"),
    risco_medio=("risco", "mean"),
    risco_max=("risco", "max")
).round(2)

print(resumo)

# GRÁFICO - RISCO NO TEMPO
plt.figure(figsize=(11, 4))

for e in estufas:
    parte = df[df["estufa"] == e]
    plt.plot(parte["data_hora"], parte["risco"], label=e)

plt.axhline(30, linestyle="--")
plt.axhline(60, linestyle="--")
plt.title("Índice de Risco ao Longo do Tempo")
plt.xlabel("Tempo")
plt.ylabel("Risco (0-100)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# GRÁFICO - RISCO MÉDIO
plt.figure(figsize=(7, 4))
plt.bar(resumo.index, resumo["risco_medio"])
plt.title("Risco Médio por Estufa")
plt.xlabel("Estufa")
plt.ylabel("Risco médio")
plt.grid(axis="y", alpha=0.3)
plt.show()

# GRÁFICO - TEMPERATURA x UMIDADE
plt.figure(figsize=(8, 5))

for e in estufas:
    parte = df[df["estufa"] == e]
    plt.scatter(
        parte["temperatura"],
        parte["umidade"],
        alpha=0.65,
        label=e
    )

plt.axvline(30, linestyle="--")
plt.axhline(55, linestyle="--")
plt.title("Temperatura x Umidade")
plt.xlabel("Temperatura (°C)")
plt.ylabel("Umidade (%)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# ESTUFA MAIS CRÍTICA
critica = resumo["risco_medio"].idxmax()

print(f"\nEstufa mais crítica: {critica}")
print(f"Risco médio: {resumo.loc[critica, 'risco_medio']}")
print(f"Risco máximo: {resumo.loc[critica, 'risco_max']}")